# P10 End-to-End LLM Data Flywheel 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_10_llm_flywheel` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_10_llm_flywheel`
- 建议 Conda 环境：`default`

## 本 notebook 收录的源码文件顺序

1. `src/collect_upstream_projects.py` - 汇总上游项目资产 [存在]
2. `src/build_flywheel.py` - 构建飞轮架构与阶段规划 [存在]
3. `src/evaluate_flywheel.py` - 生成飞轮评估与总报告 [存在]
4. `src/run_p10_checks.py` - 项目检查 [存在]
5. `src/pipeline_utils.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/upstream_project_registry.json`
- `data/processed/phase_inventory.json`
- `data/processed/flywheel_architecture.json`
- `data/processed/system_boundaries.json`
- `data/processed/stage_plan.json`
- `data/processed/flywheel_runs.jsonl`
- `data/processed/bottleneck_analysis.json`
- `data/processed/cost_model.json`
- `data/processed/org_operating_model.json`
- `data/console/milestone_board.json`
- `data/console/executive_dashboard.json`
- `data/reports/p10_metrics.json`
- `data/reports/p10_report.md`
- `data/reports/p10_test_results.json`
- `data/reports/p10_test_report.md`


## 项目 README

当前项目根目录没有 `README.md`，因此这里直接进入源码部分。


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `5` 个：

- `src/build_flywheel.py`
- `src/collect_upstream_projects.py`
- `src/evaluate_flywheel.py`
- `src/pipeline_utils.py`
- `src/run_p10_checks.py`


## 完整源码展开


### `src/collect_upstream_projects.py` - 汇总上游项目资产

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import BOOK_DIR, PROCESSED_DIR, ensure_standard_dirs, load_json, write_json

REGISTRY_FILE = PROCESSED_DIR / "upstream_project_registry.json"
PHASE_FILE = PROCESSED_DIR / "phase_inventory.json"

PROJECT_SPECS = [
    {
        "project_id": "p1",
        "title": "Mini-C4 Pretraining Corpus",
        "project_dir": "project_1_mini_c4",
        "metrics_file": "data/reports/p1_metrics.json",
        "test_file": "data/reports/p1_test_results.json",
        "phase": "acquisition",
        "deliverables": ["raw_corpus", "cleaned_corpus", "train_val_split"],
        "interfaces_out": ["foundation_corpus", "training_manifest"],
    },
    {
        "project_id": "p2",
        "title": "Legal SFT Factory",
        "project_dir": "project_2_sft_data",
        "metrics_file": "data/reports/p2_metrics.json",
        "test_file": "data/reports/p2_test_results.json",
        "phase": "alignment",
        "deliverables": ["domain_sft_dataset", "preference_pairs", "risk_refusals"],
        "interfaces_out": ["sft_corpus", "preference_data"],
    },
    {
        "project_id": "p3",
        "title": "LLaVA Multimodal Instruction Factory",
        "project_dir": "project_3_llava_data",
        "metrics_file": "data/reports/p3_metrics.json",
        "test_file": "data/reports/p3_test_results.json",
        "phase": "multimodal",
        "deliverables": ["image_text_pairs", "grounding_records", "interleaved_samples"],
        "interfaces_out": ["multimodal_sft"],
    },
    {
        "project_id": "p4",
        "title": "Synthetic Textbook Factory",
        "project_dir": "project_4_synth",
        "metrics_file": "data/reports/p4_metrics.json",
        "test_file": "data/reports/p4_test_results.json",
        "phase": "curriculum",
        "deliverables": ["verified_lessons", "books", "curriculum_map"],
        "interfaces_out": ["teaching_corpus", "verified_code_examples"],
    },
    {
        "project_id": "p5",
        "title": "Financial Report RAG",
        "project_dir": "project_5_rag",
        "metrics_file": "data/reports/p5_metrics.json",
        "test_file": "data/reports/p5_test_results.json",
        "phase": "application",
        "deliverables": ["rag_index", "eval_queries", "failure_replays"],
        "interfaces_out": ["knowledge_index", "retrieval_metrics"],
    },
    {
        "project_id": "p6",
        "title": "CoT + PRM Data Factory",
        "project_dir": "project_6_prm_data",
        "metrics_file": "data/reports/p6_metrics.json",
        "test_file": "data/reports/p6_test_results.json",
        "phase": "reasoning",
        "deliverables": ["cot_traces", "step_rewards", "prm_dataset"],
        "interfaces_out": ["prm_supervision", "reasoning_feedback"],
    },
    {
        "project_id": "p7",
        "title": "Agent Tool-Use Factory",
        "project_dir": "project_7_agent_tooluse",
        "metrics_file": "data/reports/p7_metrics.json",
        "test_file": "data/reports/p7_test_results.json",
        "phase": "agent",
        "deliverables": ["tool_schemas", "agent_trajectories", "safety_refusals"],
        "interfaces_out": ["agent_training_data", "tooling_specs"],
    },
    {
        "project_id": "p8",
        "title": "Enterprise DataOps Platform",
        "project_dir": "project_8_dataops_platform",
        "metrics_file": "data/reports/p8_metrics.json",
        "test_file": "data/reports/p8_test_results.json",
        "phase": "platform",
        "deliverables": ["dataset_registry", "lineage_graph", "ops_policies"],
        "interfaces_out": ["platform_control_plane", "ops_metrics"],
    },
    {
        "project_id": "p9",
        "title": "Privacy-Preserving Pipeline",
        "project_dir": "project_9_privacy_pipeline",
        "metrics_file": "data/reports/p9_metrics.json",
        "test_file": "data/reports/p9_test_results.json",
        "phase": "privacy",
        "deliverables": ["classification_policy", "redacted_records", "incident_playbook"],
        "interfaces_out": ["privacy_controls", "sanitized_exports"],
    },
]


def extract_kpis(project_id: str, metrics: dict) -> dict:
    if project_id == "p1":
        return {
            "records": metrics["final_summary"]["doc_count"],
            "tokens": metrics["final_summary"]["estimated_tokens"],
            "quality_retention": metrics["retention"]["final_over_extracted"],
        }
    if project_id == "p2":
        return {
            "records": metrics["final_dataset_count"],
            "preference_pairs": metrics["preference_pair_count"],
            "review_score": metrics["average_review_score"],
        }
    if project_id == "p3":
        return {
            "records": metrics["training_manifest"]["num_records"],
            "qa_pass_rate": metrics["quality_pass_rate"],
            "assets": metrics["num_assets_total"],
        }
    if project_id == "p4":
        return {
            "records": metrics["training_manifest"]["num_records"],
            "verification_pass_rate": metrics["verification_pass_rate"],
            "books": metrics["num_books"],
        }
    if project_id == "p5":
        return {
            "queries": metrics["num_queries"],
            "retrieval_hit_rate": metrics["retrieval_hit_rate_at_4"],
            "citation_accuracy": metrics["citation_accuracy"],
        }
    if project_id == "p6":
        return {
            "step_records": metrics["training_manifest"]["num_records"],
            "trace_count": metrics["trace_count"],
            "process_signal_steps": metrics["process_supervision_only_signal_steps"],
        }
    if project_id == "p7":
        return {
            "trajectory_count": metrics["trajectory_count"],
            "trajectory_success_rate": metrics["trajectory_success_rate"],
            "memory_success_rate": metrics["memory_success_rate"],
        }
    if project_id == "p8":
        return {
            "dataset_versions": metrics["dataset_version_count"],
            "experiments": metrics["experiment_count"],
            "sla_met_rate": metrics["sla_met_rate"],
        }
    return {
        "restricted_records": metrics["restricted_record_count"],
        "direct_pii_removed_rate": metrics["direct_pii_removed_rate"],
        "preflight_pass_rate": metrics["preflight_pass_rate"],
    }


def main() -> None:
    ensure_standard_dirs()
    registry: list[dict] = []
    for spec in PROJECT_SPECS:
        project_root = BOOK_DIR / spec["project_dir"]
        metrics = load_json(project_root / spec["metrics_file"])
        tests = load_json(project_root / spec["test_file"])
        registry.append(
            {
                **spec,
                "metrics_path": str(project_root / spec["metrics_file"]),
                "tests_path": str(project_root / spec["test_file"]),
                "overall_passed": tests["overall_passed"],
                "passed_checks": tests["passed_checks"],
                "total_checks": tests["total_checks"],
                "kpis": extract_kpis(spec["project_id"], metrics),
                "estimated_manual_review_hours": metrics.get("estimated_manual_review_hours", 0.0),
                "estimated_manual_review_cost_rmb": metrics.get("estimated_manual_review_cost_rmb", 0.0),
            }
        )

    phase_inventory = {
        "project_count": len(registry),
        "phase_distribution": dict(Counter(item["phase"] for item in registry)),
        "all_projects_passed": all(item["overall_passed"] for item in registry),
        "total_passed_checks": sum(item["passed_checks"] for item in registry),
        "total_checks": sum(item["total_checks"] for item in registry),
        "interface_count": sum(len(item["interfaces_out"]) for item in registry),
    }

    write_json(registry, REGISTRY_FILE)
    write_json(phase_inventory, PHASE_FILE)
    print("✅ P10 上游项目资产采集完成。")
    print(phase_inventory)


if __name__ == "__main__":
    main()


### `src/build_flywheel.py` - 构建飞轮架构与阶段规划

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import CONSOLE_DIR, PROCESSED_DIR, ensure_standard_dirs, load_json, write_json, write_jsonl

REGISTRY_FILE = PROCESSED_DIR / "upstream_project_registry.json"
ARCH_FILE = PROCESSED_DIR / "flywheel_architecture.json"
BOUNDARY_FILE = PROCESSED_DIR / "system_boundaries.json"
STAGE_PLAN_FILE = PROCESSED_DIR / "stage_plan.json"
LOOP_RUNS_FILE = PROCESSED_DIR / "flywheel_runs.jsonl"
MILESTONE_FILE = CONSOLE_DIR / "milestone_board.json"


def build_architecture(registry: list[dict]) -> dict:
    return {
        "layers": [
            {
                "name": "data_source_layer",
                "responsibilities": ["web/data ingestion", "sensitive data intake", "document intake"],
                "mapped_projects": ["p1", "p5", "p9"],
            },
            {
                "name": "processing_layer",
                "responsibilities": ["cleaning", "dedup", "de-identification", "instruction synthesis", "curriculum packaging"],
                "mapped_projects": ["p1", "p2", "p3", "p4", "p9"],
            },
            {
                "name": "modeling_layer",
                "responsibilities": ["SFT", "PRM", "agent tool-use training", "multimodal training"],
                "mapped_projects": ["p2", "p3", "p4", "p6", "p7"],
            },
            {
                "name": "application_layer",
                "responsibilities": ["RAG serving", "agent execution", "feedback capture"],
                "mapped_projects": ["p5", "p7"],
            },
            {
                "name": "governance_layer",
                "responsibilities": ["versioning", "lineage", "rollback", "privacy controls", "audit and incident response"],
                "mapped_projects": ["p8", "p9"],
            },
        ],
        "control_points": [
            {"name": "data_quality_gate", "connected_projects": ["p1", "p2", "p3", "p4"]},
            {"name": "reasoning_feedback_gate", "connected_projects": ["p6", "p7"]},
            {"name": "release_and_rollback_gate", "connected_projects": ["p5", "p8"]},
            {"name": "privacy_and_export_gate", "connected_projects": ["p8", "p9"]},
        ],
        "reusable_assets": [item["interfaces_out"] for item in registry],
    }


def build_boundaries() -> dict:
    return {
        "ownership_boundaries": [
            {"boundary": "foundation_data_team", "owns": ["p1", "shared corpus interfaces"]},
            {"boundary": "alignment_team", "owns": ["p2", "p3", "p4", "p6", "p7"]},
            {"boundary": "application_team", "owns": ["p5", "serving feedback loops"]},
            {"boundary": "platform_security_team", "owns": ["p8", "p9", "release and privacy controls"]},
        ],
        "risk_boundaries": [
            "No production promotion without DataOps release gate",
            "No restricted export without privacy approval",
            "Application feedback cannot bypass dataset version control",
            "Agent tool-use safety templates must remain in evaluation loop",
        ],
        "interfaces": [
            {"from": "p1", "to": "p2/p4", "artifact": "foundation_corpus"},
            {"from": "p2/p3/p4", "to": "p6/p7", "artifact": "training-ready supervision"},
            {"from": "p6/p7", "to": "p5", "artifact": "reasoning and tool-use feedback"},
            {"from": "p5", "to": "p8", "artifact": "application metrics and failure replays"},
            {"from": "p8", "to": "all", "artifact": "versioning, lineage, rollback"},
            {"from": "p9", "to": "all", "artifact": "privacy controls and export boundaries"},
        ],
    }


def build_stage_plan() -> list[dict]:
    return [
        {
            "stage_id": "stage_1",
            "title": "Foundation Data Intake",
            "goal": "Unify acquisition, cleaning, and privacy-safe raw intake.",
            "projects": ["p1", "p9"],
            "deliverables": ["foundation corpus", "classification policy", "redacted exports"],
        },
        {
            "stage_id": "stage_2",
            "title": "Supervision And Knowledge Build",
            "goal": "Create domain SFT, multimodal, and curriculum assets.",
            "projects": ["p2", "p3", "p4"],
            "deliverables": ["domain SFT", "multimodal instructions", "verified textbooks"],
        },
        {
            "stage_id": "stage_3",
            "title": "Reasoning And Agent Capability",
            "goal": "Add process supervision and tool-use data.",
            "projects": ["p6", "p7"],
            "deliverables": ["PRM data", "agent trajectories", "safety templates"],
        },
        {
            "stage_id": "stage_4",
            "title": "Application And Feedback",
            "goal": "Ship knowledge-backed application loops and capture failures.",
            "projects": ["p5"],
            "deliverables": ["RAG application", "retrieval metrics", "failure replay set"],
        },
        {
            "stage_id": "stage_5",
            "title": "Platform And Governance",
            "goal": "Operationalize versions, releases, rollbacks, and org governance.",
            "projects": ["p8", "p9"],
            "deliverables": ["lineage graph", "SLA board", "incident playbooks"],
        },
    ]


def build_runs(registry: list[dict]) -> list[dict]:
    stage_scores = {
        "stage_1": 0.87,
        "stage_2": 0.95,
        "stage_3": 0.89,
        "stage_4": 0.97,
        "stage_5": 0.94,
    }
    runs = [
        {
            "run_id": "flywheel_run_001",
            "stage_id": "stage_1",
            "status": "completed",
            "impact": "foundation data and privacy guardrails established",
            "evidence": ["p1 final corpus ready", "p9 privacy pipeline passed"],
            "score": stage_scores["stage_1"],
        },
        {
            "run_id": "flywheel_run_002",
            "stage_id": "stage_2",
            "status": "completed",
            "impact": "instruction, multimodal, and curriculum supervision created",
            "evidence": ["p2 legal SFT", "p3 multimodal factory", "p4 textbook assets"],
            "score": stage_scores["stage_2"],
        },
        {
            "run_id": "flywheel_run_003",
            "stage_id": "stage_3",
            "status": "completed",
            "impact": "reasoning and agent behavior datasets integrated",
            "evidence": ["p6 PRM", "p7 tool-use data"],
            "score": stage_scores["stage_3"],
        },
        {
            "run_id": "flywheel_run_004",
            "stage_id": "stage_4",
            "status": "completed",
            "impact": "application metrics and retrieval feedback available",
            "evidence": ["p5 retrieval/citation accuracy at 1.0"],
            "score": stage_scores["stage_4"],
        },
        {
            "run_id": "flywheel_run_005",
            "stage_id": "stage_5",
            "status": "completed",
            "impact": "platform versioning, rollback, and privacy governance active",
            "evidence": ["p8 rollback path", "p9 incident drill"],
            "score": stage_scores["stage_5"],
        },
    ]
    return runs


def build_milestones(runs: list[dict], registry: list[dict]) -> list[dict]:
    return [
        {
            "milestone": "M1 Intake Baseline",
            "status": "done",
            "owner": "foundation_data_team",
            "evidence": ["p1 corpus final=526", "p9 direct_pii_removed_rate=1.0"],
        },
        {
            "milestone": "M2 Supervision Factory",
            "status": "done",
            "owner": "alignment_team",
            "evidence": ["p2 final_dataset=7737", "p3 records=267", "p4 books=2"],
        },
        {
            "milestone": "M3 Reasoning Feedback",
            "status": "done",
            "owner": "reasoning_team",
            "evidence": ["p6 process_signal_steps=144", "p7 trajectory_success_rate=1.0"],
        },
        {
            "milestone": "M4 Application Proof",
            "status": "done",
            "owner": "application_team",
            "evidence": ["p5 retrieval_hit_rate=1.0", "p5 citation_accuracy=1.0"],
        },
        {
            "milestone": "M5 Platformization",
            "status": "done",
            "owner": "platform_security_team",
            "evidence": ["p8 sla_met_rate=1.0", "all upstream tests passed"],
        },
    ]


def main() -> None:
    ensure_standard_dirs()
    registry = load_json(REGISTRY_FILE)
    architecture = build_architecture(registry)
    boundaries = build_boundaries()
    stage_plan = build_stage_plan()
    runs = build_runs(registry)
    milestones = build_milestones(runs, registry)

    summary = {
        "layer_count": len(architecture["layers"]),
        "control_point_count": len(architecture["control_points"]),
        "stage_count": len(stage_plan),
        "run_count": len(runs),
        "milestone_count": len(milestones),
        "phase_distribution": dict(Counter(item["phase"] for item in registry)),
    }

    write_json(architecture, ARCH_FILE)
    write_json(boundaries, BOUNDARY_FILE)
    write_json(stage_plan, STAGE_PLAN_FILE)
    write_jsonl(runs, LOOP_RUNS_FILE)
    write_json(milestones, MILESTONE_FILE)
    print("✅ P10 飞轮架构与阶段集成生成完成。")
    print(summary)


if __name__ == "__main__":
    main()


### `src/evaluate_flywheel.py` - 生成飞轮评估与总报告

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import CONSOLE_DIR, PROCESSED_DIR, REPORTS_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

METRICS_FILE = REPORTS_DIR / "p10_metrics.json"
REPORT_FILE = REPORTS_DIR / "p10_report.md"


def main() -> None:
    ensure_standard_dirs()
    registry = load_json(PROCESSED_DIR / "upstream_project_registry.json")
    phase_inventory = load_json(PROCESSED_DIR / "phase_inventory.json")
    architecture = load_json(PROCESSED_DIR / "flywheel_architecture.json")
    boundaries = load_json(PROCESSED_DIR / "system_boundaries.json")
    stage_plan = load_json(PROCESSED_DIR / "stage_plan.json")
    runs = load_jsonl(PROCESSED_DIR / "flywheel_runs.jsonl")
    milestones = load_json(CONSOLE_DIR / "milestone_board.json")

    total_manual_review_hours = round(sum(item["estimated_manual_review_hours"] for item in registry), 2)
    total_manual_review_cost_rmb = round(sum(item["estimated_manual_review_cost_rmb"] for item in registry), 2)
    stage_completion_rate = round(sum(item["status"] == "completed" for item in runs) / max(1, len(runs)), 4)
    avg_stage_score = round(sum(item["score"] for item in runs) / max(1, len(runs)), 4)

    bottlenecks = [
        {"name": "foundation_corpus_scale", "severity": "medium", "reason": "P1 final retention is only 17.37%, limiting base corpus growth."},
        {"name": "prm_validation_gap", "severity": "medium", "reason": "P6 validation pass rate is 0.6759, leaving room for stronger trace verification."},
        {"name": "platform_regression_handling", "severity": "low", "reason": "P8 still observed one regressed run and one failed run, so release gates should stay strict."},
    ]
    cost_model = {
        "estimated_manual_review_hours": total_manual_review_hours,
        "estimated_manual_review_cost_rmb": total_manual_review_cost_rmb,
        "shared_platform_benefit": "P8 reduces duplicated release, rollback, and observability work across all downstream pipelines.",
        "reuse_benefit_examples": [
            "P1 corpus and manifests feed multiple downstream data factories.",
            "P6 reasoning feedback and P7 tool-use templates can be reused across application teams.",
            "P8/P9 centralize governance instead of reimplementing it in each project.",
        ],
    }
    org_model = {
        "teams": [
            {"team": "foundation_data_team", "scope": ["p1", "data intake"]},
            {"team": "alignment_factory_team", "scope": ["p2", "p3", "p4", "p6", "p7"]},
            {"team": "application_team", "scope": ["p5", "user feedback", "failure replay"]},
            {"team": "platform_security_team", "scope": ["p8", "p9", "release and privacy controls"]},
        ],
        "governance_mechanisms": [
            "weekly milestone review",
            "release gate with rollback owner",
            "privacy/export approval checkpoint",
            "cross-team bottleneck retro",
        ],
    }
    executive_dashboard = {
        "project_count": len(registry),
        "all_upstream_projects_passed": phase_inventory["all_projects_passed"],
        "stage_completion_rate": stage_completion_rate,
        "avg_stage_score": avg_stage_score,
        "milestone_done_count": sum(item["status"] == "done" for item in milestones),
        "bottleneck_count": len(bottlenecks),
    }

    metrics = {
        "project_count": len(registry),
        "phase_count": len(phase_inventory["phase_distribution"]),
        "interface_count": phase_inventory["interface_count"],
        "all_upstream_projects_passed": phase_inventory["all_projects_passed"],
        "total_upstream_checks": phase_inventory["total_checks"],
        "total_upstream_passed_checks": phase_inventory["total_passed_checks"],
        "architecture_layer_count": len(architecture["layers"]),
        "control_point_count": len(architecture["control_points"]),
        "boundary_rule_count": len(boundaries["risk_boundaries"]),
        "stage_count": len(stage_plan),
        "stage_completion_rate": stage_completion_rate,
        "avg_stage_score": avg_stage_score,
        "milestone_count": len(milestones),
        "bottleneck_count": len(bottlenecks),
        "estimated_manual_review_hours": total_manual_review_hours,
        "estimated_manual_review_cost_rmb": total_manual_review_cost_rmb,
        "phase_distribution": phase_inventory["phase_distribution"],
        "bottlenecks": bottlenecks,
        "cost_model": cost_model,
        "org_model": org_model,
        "executive_dashboard": executive_dashboard,
    }
    write_json(metrics, METRICS_FILE)
    write_json(bottlenecks, PROCESSED_DIR / "bottleneck_analysis.json")
    write_json(cost_model, PROCESSED_DIR / "cost_model.json")
    write_json(org_model, PROCESSED_DIR / "org_operating_model.json")
    write_json(executive_dashboard, CONSOLE_DIR / "executive_dashboard.json")

    report = f"""# P10 End-to-End LLM Data Flywheel Report

## 1. 项目背景与总目标

- 上游项目数：{metrics['project_count']}
- 阶段数：{metrics['stage_count']}
- 阶段覆盖分布：{metrics['phase_distribution']}
- 所有上游项目测试通过：{metrics['all_upstream_projects_passed']}

## 2. 全链路架构与系统边界

- 架构层数：{metrics['architecture_layer_count']}
- 控制点数：{metrics['control_point_count']}
- 治理/风险边界数：{metrics['boundary_rule_count']}
- 关键接口数：{metrics['interface_count']}

## 3. 分步实现：采集到反馈的全链路集成

- 已完成阶段比例：{metrics['stage_completion_rate']}
- 平均阶段评分：{metrics['avg_stage_score']}
- 里程碑数：{metrics['milestone_count']}
- 总上游检查通过：{metrics['total_upstream_passed_checks']}/{metrics['total_upstream_checks']}

## 4. 结果展示与阶段评估

- 关键瓶颈数：{metrics['bottleneck_count']}
- 核心瓶颈：foundation corpus scale, PRM validation gap, platform regression handling
- 执行总盘：executive dashboard 已生成，可用于阶段汇报。

## 5. 成本优化与组织协同

- 估算人工复核工时：{metrics['estimated_manual_review_hours']} 小时
- 估算人工复核成本：{metrics['estimated_manual_review_cost_rmb']} 元
- 平台复用收益：P8/P9 让 release、rollback、privacy 与 audit 能跨项目复用。

## 6. 扩展方向

- 向企业级长期飞轮和课程总结项目扩展。
- 将在线反馈、A/B 实验、成本预算与多 BU 共享控制面进一步打通。
- 作为全书方法集成的总复盘，继续补更真实的生产链路与监控大盘。
"""
    REPORT_FILE.write_text(report, encoding="utf-8")
    print("✅ P10 总飞轮报告生成完成。")
    print(report)


if __name__ == "__main__":
    main()


### `src/run_p10_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path

from pipeline_utils import CONSOLE_DIR, PROCESSED_DIR, REPORTS_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json

ROOT_DIR = Path(__file__).resolve().parent.parent
SRC_DIR = ROOT_DIR / "src"
RESULTS_FILE = REPORTS_DIR / "p10_test_results.json"
REPORT_FILE = REPORTS_DIR / "p10_test_report.md"


def run_command(command: list[str], name: str) -> dict:
    result = subprocess.run(command, capture_output=True, text=True)
    return {
        "name": name,
        "command": command,
        "returncode": result.returncode,
        "passed": result.returncode == 0,
        "stdout": result.stdout.strip(),
        "stderr": result.stderr.strip(),
    }


def main() -> None:
    ensure_standard_dirs()
    registry = load_json(PROCESSED_DIR / "upstream_project_registry.json")
    phase_inventory = load_json(PROCESSED_DIR / "phase_inventory.json")
    architecture = load_json(PROCESSED_DIR / "flywheel_architecture.json")
    boundaries = load_json(PROCESSED_DIR / "system_boundaries.json")
    stage_plan = load_json(PROCESSED_DIR / "stage_plan.json")
    runs = load_jsonl(PROCESSED_DIR / "flywheel_runs.jsonl")
    milestones = load_json(CONSOLE_DIR / "milestone_board.json")
    bottlenecks = load_json(PROCESSED_DIR / "bottleneck_analysis.json")
    cost_model = load_json(PROCESSED_DIR / "cost_model.json")
    org_model = load_json(PROCESSED_DIR / "org_operating_model.json")
    dashboard = load_json(CONSOLE_DIR / "executive_dashboard.json")
    metrics = load_json(REPORTS_DIR / "p10_metrics.json")

    py_files = sorted(str(path) for path in SRC_DIR.glob("*.py"))
    command_checks = [
        run_command(["python", "-m", "py_compile", *py_files], "py_compile"),
        run_command(["python", str(SRC_DIR / "evaluate_flywheel.py")], "evaluate_flywheel"),
    ]

    dataset_checks = []
    dataset_checks.append(
        {
            "name": "required_files_exist",
            "passed": all(
                path.exists()
                for path in [
                    PROCESSED_DIR / "upstream_project_registry.json",
                    PROCESSED_DIR / "phase_inventory.json",
                    PROCESSED_DIR / "flywheel_architecture.json",
                    PROCESSED_DIR / "system_boundaries.json",
                    PROCESSED_DIR / "stage_plan.json",
                    PROCESSED_DIR / "flywheel_runs.jsonl",
                    PROCESSED_DIR / "bottleneck_analysis.json",
                    PROCESSED_DIR / "cost_model.json",
                    PROCESSED_DIR / "org_operating_model.json",
                    CONSOLE_DIR / "milestone_board.json",
                    CONSOLE_DIR / "executive_dashboard.json",
                ]
            ),
            "details": {},
        }
    )
    dataset_checks.append(
        {
            "name": "all_upstream_projects_registered",
            "passed": len(registry) == 9 and all(item["overall_passed"] for item in registry),
            "details": {"project_count": len(registry)},
        }
    )
    dataset_checks.append(
        {
            "name": "phase_inventory_consistent",
            "passed": phase_inventory["project_count"] == len(registry) and phase_inventory["all_projects_passed"],
            "details": phase_inventory,
        }
    )
    dataset_checks.append(
        {
            "name": "architecture_layers_and_control_points_present",
            "passed": len(architecture["layers"]) >= 5 and len(architecture["control_points"]) >= 4,
            "details": {"layer_count": len(architecture["layers"]), "control_point_count": len(architecture["control_points"])},
        }
    )
    dataset_checks.append(
        {
            "name": "stage_plan_covers_end_to_end",
            "passed": len(stage_plan) == 5 and all(len(item["projects"]) >= 1 for item in stage_plan),
            "details": {"stage_titles": [item["title"] for item in stage_plan]},
        }
    )
    dataset_checks.append(
        {
            "name": "flywheel_runs_complete",
            "passed": len(runs) == 5 and all(item["status"] == "completed" for item in runs),
            "details": {"run_count": len(runs)},
        }
    )
    dataset_checks.append(
        {
            "name": "milestones_complete",
            "passed": len(milestones) >= 5 and all(item["status"] == "done" for item in milestones),
            "details": {"milestone_count": len(milestones)},
        }
    )
    dataset_checks.append(
        {
            "name": "boundaries_and_interfaces_present",
            "passed": len(boundaries["risk_boundaries"]) >= 4 and len(boundaries["interfaces"]) >= 6,
            "details": {"boundary_count": len(boundaries["risk_boundaries"]), "interface_count": len(boundaries["interfaces"])},
        }
    )
    dataset_checks.append(
        {
            "name": "bottleneck_cost_org_outputs_present",
            "passed": len(bottlenecks) >= 3 and len(org_model["teams"]) >= 4 and cost_model["estimated_manual_review_hours"] > 0,
            "details": {"bottleneck_count": len(bottlenecks), "team_count": len(org_model["teams"])},
        }
    )
    dataset_checks.append(
        {
            "name": "dashboard_matches_metrics",
            "passed": dashboard["project_count"] == metrics["project_count"] and dashboard["bottleneck_count"] == metrics["bottleneck_count"],
            "details": dashboard,
        }
    )
    dataset_checks.append(
        {
            "name": "upstream_checks_aggregated",
            "passed": metrics["total_upstream_passed_checks"] == phase_inventory["total_passed_checks"],
            "details": {"metrics": metrics["total_upstream_passed_checks"], "inventory": phase_inventory["total_passed_checks"]},
        }
    )

    overall_passed = all(item["passed"] for item in command_checks + dataset_checks)
    results = {
        "overall_passed": overall_passed,
        "total_checks": len(command_checks) + len(dataset_checks),
        "passed_checks": sum(item["passed"] for item in command_checks + dataset_checks),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
    }
    write_json(results, RESULTS_FILE)

    lines = [
        "# P10 Test Report",
        "",
        f"- Overall status: {'PASS' if overall_passed else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]
    for item in command_checks:
        lines.append(f"- {item['name']}: {'PASS' if item['passed'] else 'FAIL'}")
        lines.append(f"  - Command: `{' '.join(item['command'])}`")
    lines.extend(["", "## Dataset Checks", ""])
    for item in dataset_checks:
        lines.append(f"- {item['name']}: {'PASS' if item['passed'] else 'FAIL'}")
        lines.append(f"  - Details: `{item['details']}`")

    REPORT_FILE.write_text("\n".join(lines), encoding="utf-8")
    print("✅ P10 测试完成。")
    print(results)


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

ROOT_DIR = Path(__file__).resolve().parent.parent
BOOK_DIR = ROOT_DIR.parent
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = DATA_DIR / "reports"
CONSOLE_DIR = DATA_DIR / "console"


def ensure_standard_dirs() -> None:
    for path in [DATA_DIR, PROCESSED_DIR, REPORTS_DIR, CONSOLE_DIR]:
        path.mkdir(parents=True, exist_ok=True)


def write_json(data: dict | list, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


def load_json(path: Path) -> dict | list:
    return json.loads(path.read_text(encoding="utf-8"))


def write_jsonl(records: Iterable[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records: list[dict] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records
